# 生成式对话机器人

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("./alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-389m-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-389m-zh', vocab_size=42437, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [5]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [7]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [8]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [9]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-389m-zh")

## Step5 配置训练参数

In [10]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=2
)

## Step6 创建训练器

In [11]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

## Step7 模型训练

In [12]:
trainer.train()

  0%|          | 0/3356 [00:00<?, ?it/s]

{'loss': 3.2787, 'grad_norm': 106.77286529541016, 'learning_rate': 4.9851013110846245e-05, 'epoch': 0.01}
{'loss': 3.4583, 'grad_norm': 103.0752944946289, 'learning_rate': 4.9702026221692494e-05, 'epoch': 0.01}
{'loss': 3.4951, 'grad_norm': 84.02734375, 'learning_rate': 4.9553039332538736e-05, 'epoch': 0.02}
{'loss': 3.414, 'grad_norm': 93.71533203125, 'learning_rate': 4.9404052443384986e-05, 'epoch': 0.02}
{'loss': 3.2154, 'grad_norm': 97.94625091552734, 'learning_rate': 4.925506555423123e-05, 'epoch': 0.03}
{'loss': 3.2269, 'grad_norm': 83.81385040283203, 'learning_rate': 4.910607866507748e-05, 'epoch': 0.04}
{'loss': 3.18, 'grad_norm': 101.79875183105469, 'learning_rate': 4.895709177592372e-05, 'epoch': 0.04}
{'loss': 3.0686, 'grad_norm': 57.5830078125, 'learning_rate': 4.880810488676997e-05, 'epoch': 0.05}
{'loss': 2.9377, 'grad_norm': 36.75483703613281, 'learning_rate': 4.865911799761621e-05, 'epoch': 0.05}
{'loss': 3.0856, 'grad_norm': 20.232322692871094, 'learning_rate': 4.85101

TrainOutput(global_step=3356, training_loss=2.154776271108507, metrics={'train_runtime': 3500.4724, 'train_samples_per_second': 15.345, 'train_steps_per_second': 0.959, 'total_flos': 9562917601247232.0, 'train_loss': 2.154776271108507, 'epoch': 1.9992553429145878})

## Step8 模型推理

In [13]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [14]:
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(ipt, max_length=256, do_sample=True, )

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': 'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧是指解决某个具体题目或某个考试中的特定问题的一种有效的方法。以下是一些常用的考试技巧：\n\n1. 利用分数的表来分类题目。 在分析题目过程中，应该先通过对题目的分类进行理解，比如将题目划分为A，B，C三个类别，这样可以有效区分出哪些为真题，哪些为假题。\n2. 运用加减法的结构和公式。 在解题前，应该知道考试中的题目加减法是什么，它是一个加减法公式，可以通过简化运算来求解。\n3. 形成良好的作息时间。 在放松或休息时做一些放松题，可以帮助大脑放松，增强记忆能力。\n4. 坚持练习。 在考试之前，应该制定好复习计划，通过反复练习来巩固知识。'}]

In [15]:
ipt = "Human: {}\n{}".format("几点睡觉最好？", "").strip() + "\n\nAssistant: "
pipe(ipt, max_length=256, do_sample=True, )

[{'generated_text': 'Human: 几点睡觉最好？\n\nAssistant: 不同地区的气候和生活方式可能会影响人们的睡眠时间。您可以询问当地医生或咨询您的睡眠专家以获得准确的建议。睡眠时间对人的身体健康非常重要，因此建议每晚在7-9点睡觉是最佳时间。'}]